# Push all three stage models to Hugging Face (from Kaggle, not your home connection)

Home upload bandwidth was the bottleneck pushing large files to Hugging Face/Kaggle -- this does the same push from inside Kaggle's datacenter network instead, which is fast and reliable (same reason the full backup zip upload worked fine earlier).

**Before running:**
1. Notebook settings -> Internet -> On (GPU not needed, this is just I/O + a CPU-only weight conversion)
2. Add-ons -> Secrets -> add a new secret named `HF_TOKEN` with your Hugging Face **write** token as the value. This keeps the token out of the notebook text entirely.

In [ ]:
!pip install -q huggingface_hub transformers

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

## Pull the full backup down and extract it
This is a Hugging Face -> Kaggle transfer, both datacenter-side, so it's fast regardless of your home connection.

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile

zip_path = hf_hub_download(repo_id="abhijit26/harry-potter-backup", repo_type="dataset", filename="harry-potter-gpt-full.zip")
with zipfile.ZipFile(zip_path) as z:
    z.extractall("/kaggle/working/extracted")
print("Extracted to /kaggle/working/extracted")

## Convert the Base (pretrain-only) nanoGPT checkpoint to HuggingFace format
SFT and DPO are already in HF format from the pipeline; only Base still needs converting.

In [ ]:
!git clone https://github.com/abhijitdalal26/harry-potter-gpt.git
%cd harry-potter-gpt/nanoGPT
!python convert_to_hf.py --ckpt_path=/kaggle/working/extracted/out-harry-potter/ckpt.pt --output_dir=/kaggle/working/harry-potter-gpt-base-hf

## Push all three as separate model repos

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

CARD = """---
license: mit
language: en
tags:
- gpt2
- harry-potter
- nanogpt
pipeline_tag: text-generation
---

# Harry Potter GPT -- {stage} stage

Part of a from-scratch pretrain -> SFT -> DPO pipeline. This checkpoint is the **{stage}** stage.

Full write-up: https://abhijitdalal.vercel.app/projects/harry-potter-gpt
Source: https://github.com/abhijitdalal26/harry-potter-gpt
"""

def push_model(local_dir, repo_id, stage):
    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True, private=False)
    api.upload_folder(
        folder_path=local_dir,
        repo_id=repo_id,
        repo_type="model",
        ignore_patterns=["checkpoint-*", "training_args.bin", "optimizer.pt", "rng_state.pth", "scheduler.pt"],
    )
    api.upload_file(
        path_or_fileobj=CARD.format(stage=stage).encode(),
        path_in_repo="README.md",
        repo_id=repo_id,
        repo_type="model",
    )
    print("Pushed", repo_id)

push_model("/kaggle/working/harry-potter-gpt-base-hf", "abhijit26/harry-potter-gpt-base", "Base (pretrain-only)")
push_model("/kaggle/working/extracted/harry-potter-hf", "abhijit26/harry-potter-gpt-sft", "SFT")
push_model("/kaggle/working/extracted/harry-potter-hf-dpo", "abhijit26/harry-potter-gpt-dpo", "DPO")